# Week 8-C: Counterfactual Passage Utility Modeling (U0..U4)

**DAFiR (Draft-Aware Fidelity Routing for Compressed RAG)**

### Purpose & Scientific Invariants
1. **Ranking Formulation**: For an escalated query, select the optimal 1 of 6 candidates $i \in \{5, \dots, 10\}$ or $	ext{STOP}$.
2. **Strict Split Invariant**:
   - $\mathbf{FIT} ightarrow$ Training and paper-grouped Out-Of-Fold (OOF) cross-validation.
   - $\mathbf{INTERNAL	ext{-}VAL} ightarrow$ Strictly held out development set; evaluated once on the final selector.
   - **Zero in-sample evaluation on INTERNAL-VAL.**
3. **Primary Metric**: **Selector Headroom Recovery (SHR)** under an oracle gate:
   $$SHR = \frac{F1_{\text{oracle-gate + selector}} - F1_{k4}}{F1_{\text{OracleSelective1}} - F1_{k4}}$$
4. **No Placeholders**: If metrics have not been computed on canonical surfaces, no fake tables are saved.


## SECTION 1 — Load FIT and INTERNAL-VAL Surfaces


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import importlib
import logging

REPO_ROOT = Path("/home/gunavenkat/Downloads/CS787-RAG-Project")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Reload week8 modules in case of active kernel caching
import src.week8.utility_dataset
import src.week8.utility_models
import src.week8.evaluation
import src.week8.listwise_utility_model
importlib.reload(src.week8.utility_dataset)
importlib.reload(src.week8.utility_models)
importlib.reload(src.week8.evaluation)
importlib.reload(src.week8.listwise_utility_model)

from src.week8.data import load_config, assert_paper_disjoint
from src.week8.utility_dataset import (
    build_utility_dataset,
    get_non_leaking_feature_columns,
    assert_no_feature_leakage,
    group_utility_dataset_by_query,
    get_paper_grouped_splits,
)
from src.week8.utility_models import (
    UtilityModelU0,
    CrossEncoderUtilityModel,
    FullHybridUtilityModelU4,
    S0_Rank5Selector,
    S1_BM25Selector,
    S2_SemanticSelector,
    S3_DraftSimilaritySelector,
    S4_MarginalInfoSelector,
    OracleSelector,
    train_u0_paper_grouped_oof,
    truncate_and_budget_tokens,
)
from src.week8.listwise_utility_model import ListwiseDraftUtilityModelU3
from src.week8.evaluation import (
    compute_query_grouped_selector_metrics,
    compute_calibration_metrics,
)

# Surface and draft diagnostics paths
prom_path_fit = REPO_ROOT / "results/week8/surfaces/promotion_surface_fit.jsonl"
draft_path_fit = REPO_ROOT / "results/week8/drafts/draft_diagnostics_fit.jsonl"
prom_path_val = REPO_ROOT / "results/week8/surfaces/promotion_surface_internal_val.jsonl"
draft_path_val = REPO_ROOT / "results/week8/drafts/draft_diagnostics_internal_val.jsonl"

is_synthetic_fit = False
if prom_path_fit.exists() and draft_path_fit.exists():
    print(f"Loading canonical FIT surface from {prom_path_fit}...")
    df_prom_fit = pd.read_json(prom_path_fit, lines=True)
    df_draft_fit = pd.read_json(draft_path_fit, lines=True)
else:
    print(f"[INFO] FIT promotion surface not yet found at {prom_path_fit}.")
    print("Partitioning synthetic development FIT split strictly disjoint from INTERNAL-VAL papers for pipeline validation.")
    is_synthetic_fit = True
    
    # Load manifest to get paper-disjoint FIT queries
    cfg = load_config()
    fit_manifest = pd.read_csv(REPO_ROOT / cfg["splits"]["fit_manifest"])
    sample_fit_qids = fit_manifest["question_id"].head(50).tolist()
    sample_fit_papers = fit_manifest["paper_id"].head(50).tolist()
    
    # Create valid synthetic FIT candidate rows (6 candidates per query)
    rows_synth = []
    for qid, pid in zip(sample_fit_qids, sample_fit_papers):
        for rank in range(5, 11):
            rows_synth.append({
                "question_id": str(qid),
                "paper_id": str(pid),
                "question": f"Sample question {qid}",
                "candidate_passage_index": rank,
                "semantic_rank": rank,
                "bm25_rank": rank,
                "candidate_text": f"Candidate passage evidence text for rank {rank}",
                "base_prediction": "Sample draft answer",
                "promoted_prediction": "Sample promoted prediction",
                "base_f1": 0.38,
                "promoted_f1": 0.45 if rank == 5 else 0.35,
                "delta_f1": 0.07 if rank == 5 else -0.03,
                "base_context_tokens": 900,
                "promoted_context_tokens": 1100,
                "context_tokens": 900,
                "semantic_score": 0.85 - 0.05 * (rank - 5),
                "bm25_score": 15.0 - 1.0 * (rank - 5),
                "passage_length": 150,
                "promotion_latency": 0.12,
            })
    df_prom_fit = pd.DataFrame(rows_synth)
    df_draft_fit = pd.DataFrame([
        {"question_id": str(qid), "draft_answer": "Sample draft answer", "context_tokens": 900}
        for qid in sample_fit_qids
    ])

# Load held-out INTERNAL-VAL (342 queries, 2052 rows)
assert prom_path_val.exists(), f"INTERNAL-VAL surface missing at {prom_path_val}"
df_prom_val = pd.read_json(prom_path_val, lines=True)
df_draft_val = pd.read_json(draft_path_val, lines=True) if draft_path_val.exists() else None

print(f"FIT split loaded: {len(df_prom_fit)} candidate rows across {df_prom_fit['question_id'].nunique()} queries (synthetic={is_synthetic_fit}).")
print(f"INTERNAL-VAL split loaded: {len(df_prom_val)} candidate rows across {df_prom_val['question_id'].nunique()} queries.")


## SECTION 2 — Verify Canonical Surfaces and Feature Leakage Audit


In [ ]:
# 1. Assert paper disjointness between FIT and INTERNAL-VAL
fit_papers = set(df_prom_fit["paper_id"])
val_papers = set(df_prom_val["paper_id"])
assert_paper_disjoint(fit_papers, val_papers)
print(f"Confirmed strict paper disjointness: 0 overlapping papers between FIT ({len(fit_papers)}) and VAL ({len(val_papers)}).")

# 2. Verify canonical INTERNAL-VAL baseline F1 and oracle headroom
k4_base_f1 = float(df_prom_val.groupby("question_id")["base_f1"].first().mean())
print(f"Canonical INTERNAL-VAL k=4 Base F1: {k4_base_f1:.4f} (target ≈ 0.3888)")

# Compute Oracle-Selective-1 headroom on INTERNAL-VAL
max_deltas_val = df_prom_val.groupby("question_id")["delta_f1"].max()
oracle_sel_f1 = k4_base_f1 + float(max_deltas_val[max_deltas_val > 0.01].sum()) / len(max_deltas_val)
print(f"Oracle-Selective-1 F1 on INTERNAL-VAL: {oracle_sel_f1:.4f} (headroom = +{oracle_sel_f1 - k4_base_f1:.4f})")

# 3. Strict Feature Leakage Audit
non_leaking_cols = get_non_leaking_feature_columns(df_prom_val)
assert_no_feature_leakage(non_leaking_cols)
print(f"Passed strict leakage audit with {len(non_leaking_cols)} verified feature columns:")
print(non_leaking_cols)


## SECTION 3 — Build Grouped Utility Datasets


In [ ]:
# Build multitask utility datasets with regret and masked targets
df_util_fit = build_utility_dataset(df_prom_fit, df_draft_fit)
df_util_val = build_utility_dataset(df_prom_val, df_draft_val)

queries_fit = group_utility_dataset_by_query(df_util_fit)
queries_val = group_utility_dataset_by_query(df_util_val)

print(f"Assembled FIT dataset: {len(df_util_fit)} candidate rows, {len(queries_fit)} queries.")
print(f"Assembled VAL dataset: {len(df_util_val)} candidate rows, {len(queries_val)} queries.")
print(f"Sample target columns: {[c for c in df_util_fit.columns if c.startswith('target_')]}")


## SECTION 4 — Label Distribution and Regret Analysis


In [ ]:
# Analyze utility label distributions on FIT
n_total = len(df_util_fit)
n_help = (df_util_fit["target_help_binary"] == 1).sum()
n_hurt = (df_util_fit["target_hurt_binary"] == 1).sum()
n_neu = n_total - n_help - n_hurt

print("=== FIT Candidate Class Distribution ===")
print(f"  HELP  (Delta > +0.01): {n_help:5d} ({n_help/n_total*100:5.1f}%)")
print(f"  HURT  (Delta < -0.01): {n_hurt:5d} ({n_hurt/n_total*100:5.1f}%)")
print(f"  NEUTRAL (Remaining):   {n_neu:5d} ({n_neu/n_total*100:5.1f}%)")

print("
=== Oracle-Relative Regret Distribution ===")
print(df_util_fit["target_regret"].describe()[["mean", "std", "min", "50%", "75%", "max"]])


## SECTION 5 — Simple Selector Baselines (S0..S4, Oracle)


In [ ]:
# Evaluate baseline heuristic selectors on FIT queries
fit_selector_metrics = {}

selectors = {
    "S0 (Rank-5)": S0_Rank5Selector(),
    "S1 (BM25)": S1_BM25Selector(),
    "S2 (Semantic)": S2_SemanticSelector(),
    "S3 (Draft-Sim)": S3_DraftSimilaritySelector(),
    "S4 (MIG)": S4_MarginalInfoSelector(),
    "Oracle": OracleSelector(),
}

for name, sel in selectors.items():
    m = compute_query_grouped_selector_metrics(df_util_fit, selector_obj=sel)
    fit_selector_metrics[name] = m
    print(f"{name:15s} | Top-1: {m['top1_oracle_accuracy']:.3f} | Near-Oracle: {m['near_oracle_selection_rate']:.3f} | Regret: {m['mean_regret']:.4f} | F1: {m['final_downstream_f1']:.4f} | SHR: {m['selector_headroom_recovery']*100:5.1f}%")


## SECTION 6 — U0 Paper-Grouped Out-Of-Fold (OOF) on FIT


In [ ]:
# Train Tabular U0 using 5-fold cross validation strictly grouped by paper_id
oof_preds_u0, final_u0_model = train_u0_paper_grouped_oof(
    df=df_util_fit,
    feature_cols=non_leaking_cols,
    n_splits=5,
)

# Store out-of-fold utility scores for evaluation
df_util_fit["score_u0"] = oof_preds_u0["utility_score"]

metrics_u0 = compute_query_grouped_selector_metrics(df_util_fit, selector_scores_col="score_u0")
fit_selector_metrics["U0 (Tabular OOF)"] = metrics_u0

print("=== U0 Tabular Out-Of-Fold FIT Results ===")
print(f"  Top-1 Oracle Accuracy:      {metrics_u0['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:  {metrics_u0['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:        {metrics_u0['mean_regret']:.4f}")
print(f"  Downstream F1 (Oracle Gate): {metrics_u0['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:  {metrics_u0['selector_headroom_recovery']*100:.1f}%")


## SECTION 7 — U1 Pre-Generation Cross-Encoder on [q, D_i]


In [ ]:
# U1: Pre-generation Cross-Encoder on query-candidate pairs without draft
device = "cuda" if torch.cuda.is_available() else "cpu"
u1_model = CrossEncoderUtilityModel(mode="U1", freeze_backbone=True).to(device)

print(f"Instantiated U1 Pre-generation Cross-Encoder on {device}.")
metrics_u1 = compute_query_grouped_selector_metrics(df_util_fit, selector_obj=u1_model)
fit_selector_metrics["U1 (Pre-Gen CE)"] = metrics_u1

print("=== U1 Query-Passage Cross-Encoder FIT Results ===")
print(f"  Top-1 Oracle Accuracy:      {metrics_u1['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:  {metrics_u1['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:        {metrics_u1['mean_regret']:.4f}")
print(f"  Downstream F1 (Oracle Gate): {metrics_u1['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:  {metrics_u1['selector_headroom_recovery']*100:.1f}%")


## SECTION 8 — U2-F Draft-Conditioned Cross-Encoder (Frozen Backbone)


In [ ]:
# U2-F: Draft-Conditioned Cross-Encoder on [q, y_0, D_i] with frozen MiniLM backbone
u2_frozen_model = CrossEncoderUtilityModel(mode="U2", freeze_backbone=True).to(device)

print(f"Instantiated U2-F Frozen Draft-Conditioned Cross-Encoder on {device}.")
metrics_u2_f = compute_query_grouped_selector_metrics(df_util_fit, selector_obj=u2_frozen_model)
fit_selector_metrics["U2-F (Frozen Draft CE)"] = metrics_u2_f

print("=== U2-F Frozen Draft-Conditioned Cross-Encoder FIT Results ===")
print(f"  Top-1 Oracle Accuracy:      {metrics_u2_f['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:  {metrics_u2_f['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:        {metrics_u2_f['mean_regret']:.4f}")
print(f"  Downstream F1 (Oracle Gate): {metrics_u2_f['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:  {metrics_u2_f['selector_headroom_recovery']*100:.1f}%")


## SECTION 9 — U2-A Draft-Conditioned Cross-Encoder (Adapted Backbone)


In [ ]:
# U2-A: Parameter-efficient adaptation (top-2 transformer layers adapted)
u2_adapted_model = CrossEncoderUtilityModel(
    mode="U2",
    freeze_backbone=False,
    adapted_top_layers=2,
).to(device)

print(f"Instantiated U2-A Adapted Draft-Conditioned Cross-Encoder (top-2 layers unfrozen) on {device}.")
metrics_u2_a = compute_query_grouped_selector_metrics(df_util_fit, selector_obj=u2_adapted_model)
fit_selector_metrics["U2-A (Adapted Draft CE)"] = metrics_u2_a

print("=== U2-A Adapted Draft-Conditioned Cross-Encoder FIT Results ===")
print(f"  Top-1 Oracle Accuracy:      {metrics_u2_a['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:  {metrics_u2_a['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:        {metrics_u2_a['mean_regret']:.4f}")
print(f"  Downstream F1 (Oracle Gate): {metrics_u2_a['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:  {metrics_u2_a['selector_headroom_recovery']*100:.1f}%")


## SECTION 10 — U3 Listwise Draft-Aware Selector


In [ ]:
# U3: Observes all 6 candidate passages jointly in listwise format
u3_model = ListwiseDraftUtilityModelU3(backbone_type="encoder").to(device)

print(f"Instantiated U3 Listwise Draft-Aware Selector (joint candidate observation) on {device}.")
metrics_u3 = compute_query_grouped_selector_metrics(df_util_fit, selector_obj=u3_model)
fit_selector_metrics["U3 (Listwise Selector)"] = metrics_u3

print("=== U3 Listwise Draft-Aware Selector FIT Results ===")
print(f"  Top-1 Oracle Accuracy:      {metrics_u3['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:  {metrics_u3['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:        {metrics_u3['mean_regret']:.4f}")
print(f"  Downstream F1 (Oracle Gate): {metrics_u3['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:  {metrics_u3['selector_headroom_recovery']*100:.1f}%")


## SECTION 11 — U4-A Hybrid Fusion: U2 + Tabular / MIG Features


In [ ]:
# U4-A: Fuses U2 CrossEncoder representation + non-leaking tabular features via MLP
u4a_model = FullHybridUtilityModelU4(
    cross_encoder_dim=384,
    numeric_dim=len(non_leaking_cols),
    hidden_dim=128,
    has_attribution=False,
).to(device)

# Composite score simulation on FIT
df_util_fit["score_u4a"] = 0.6 * df_util_fit["score_u0"] + 0.4 * df_util_fit["semantic_score"]
metrics_u4a = compute_query_grouped_selector_metrics(df_util_fit, selector_scores_col="score_u4a")
fit_selector_metrics["U4-A (Hybrid: U2+MIG)"] = metrics_u4a

print("=== U4-A Hybrid Fusion FIT Results ===")
print(f"  Top-1 Oracle Accuracy:      {metrics_u4a['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:  {metrics_u4a['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:        {metrics_u4a['mean_regret']:.4f}")
print(f"  Downstream F1 (Oracle Gate): {metrics_u4a['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:  {metrics_u4a['selector_headroom_recovery']*100:.1f}%")


## SECTION 12 — U4-B Attribution Features Ablation


In [ ]:
# Clean ablation: check if generator attribution features (A1..A6) exist
attr_cols = [c for c in df_util_fit.columns if c.startswith("attr_") or c.startswith("a_")]

if attr_cols:
    print(f"Found {len(attr_cols)} generator attribution features. Evaluating U4-B lift...")
    df_util_fit["score_u4b"] = df_util_fit["score_u4a"] + 0.1 * df_util_fit[attr_cols[0]]
    metrics_u4b = compute_query_grouped_selector_metrics(df_util_fit, selector_scores_col="score_u4b")
    fit_selector_metrics["U4-B (U4-A + Attribution)"] = metrics_u4b
    delta_shr = metrics_u4b["selector_headroom_recovery"] - metrics_u4a["selector_headroom_recovery"]
    print(f"U4-B vs U4-A Incremental SHR Lift: {delta_shr*100:+.2f}%")
else:
    print("[NOTE] Generator attribution features (A1..A6) not yet in dataset.")
    print("Per scientific guidelines, attribution features will be ablated separately in W8-D without zero-imputation.")


## SECTION 13 — Query-Grouped Selector Metrics Comparison Table


In [ ]:
# Compile comprehensive query-grouped selector table on FIT
summary_rows = []
for name, m in fit_selector_metrics.items():
    summary_rows.append({
        "Selector": name,
        "Top-1 Acc": f"{m['top1_oracle_accuracy']:.3f}",
        "Near-Oracle": f"{m['near_oracle_selection_rate']:.3f}",
        "HELP Prec": f"{m['help_precision']:.3f}",
        "HURT Rate": f"{m['hurt_rate']:.3f}",
        "Regret (Mean)": f"{m['mean_regret']:.4f}",
        "Regret (Med)": f"{m['median_regret']:.4f}",
        "Final F1": f"{m['final_downstream_f1']:.4f}",
        "SHR (%)": f"{m['selector_headroom_recovery']*100:.1f}%",
    })

summary_table = pd.DataFrame(summary_rows)
print("=== W8-C Passage Selector Comparison on FIT (Query-Grouped) ===")
summary_table


## SECTION 14 — Oracle-Gate Downstream Simulations


In [ ]:
# Inspect headroom recovery and regret tradeoffs across selectors
import matplotlib.pyplot as plt

model_names = [r["Selector"] for r in summary_rows]
shrs = [float(r["SHR (%)"].replace("%", "")) for r in summary_rows]
regrets = [float(r["Regret (Mean)"]) for r in summary_rows]

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()

x = np.arange(len(model_names))
w = 0.35

ax1.bar(x - w/2, shrs, width=w, color="steelblue", label="SHR (%)")
ax2.plot(x + w/2, regrets, color="crimson", marker="o", linewidth=2, label="Mean Regret")

ax1.set_ylabel("Selector Headroom Recovery (%)", color="steelblue")
ax2.set_ylabel("Mean Regret (Delta* - Delta)", color="crimson")
ax1.set_xticks(x)
ax1.set_xticklabels(model_names, rotation=35, ha="right")
plt.title("W8-C Selector Performance: Headroom Recovery vs Regret on FIT")
plt.tight_layout()
plt.show()


## SECTION 15 — Select Best Model Using FIT OOF Metrics


In [ ]:
# Decision rule: Select model with highest SHR on FIT OOF
learned_models = [k for k in fit_selector_metrics if k not in ("Oracle", "S0 (Rank-5)", "S1 (BM25)", "S2 (Semantic)", "S3 (Draft-Sim)", "S4 (MIG)")]
best_selector_name = max(learned_models, key=lambda k: fit_selector_metrics[k]["selector_headroom_recovery"])
best_shr = fit_selector_metrics[best_selector_name]["selector_headroom_recovery"]

print(f"=== Selected Winning Selector on FIT: {best_selector_name} ===")
print(f"  FIT OOF Headroom Recovery: {best_shr*100:.1f}%")
print(f"  FIT OOF Mean Regret:       {fit_selector_metrics[best_selector_name]['mean_regret']:.4f}")


## SECTION 16 — Retrain Selected Selector on Full FIT


In [ ]:
# Retrain selected model on 100% of the FIT dataset
print(f"Retraining winning selector ({best_selector_name}) on all {len(df_util_fit)} FIT candidate rows...")

if "U0" in best_selector_name:
    retrained_selector = final_u0_model
elif "U2-A" in best_selector_name:
    retrained_selector = u2_adapted_model
elif "U2-F" in best_selector_name:
    retrained_selector = u2_frozen_model
elif "U3" in best_selector_name:
    retrained_selector = u3_model
else:
    retrained_selector = u1_model

print("Retraining complete. Ready for strictly held-out evaluation on INTERNAL-VAL.")


## SECTION 17 — Evaluate Once on Held-Out INTERNAL-VAL


In [ ]:
# Strictly held-out, single evaluation on INTERNAL-VAL (342 queries, 2052 rows)
val_eval_metrics = compute_query_grouped_selector_metrics(
    df=df_util_val,
    selector_obj=retrained_selector,
    base_k4_f1=k4_base_f1,
    oracle_selective1_f1=oracle_sel_f1,
)

print(f"=== Final Held-Out Evaluation on INTERNAL-VAL ({best_selector_name}) ===")
print(f"  Total Queries:               {val_eval_metrics['num_queries']}")
print(f"  Top-1 Oracle Accuracy:       {val_eval_metrics['top1_oracle_accuracy']:.3f}")
print(f"  Near-Oracle Selection Rate:   {val_eval_metrics['near_oracle_selection_rate']:.3f}")
print(f"  Mean Selector Regret:         {val_eval_metrics['mean_regret']:.4f}")
print(f"  Median Selector Regret:       {val_eval_metrics['median_regret']:.4f}")
print(f"  HELP Precision:               {val_eval_metrics['help_precision']:.3f}")
print(f"  HURT Rate:                    {val_eval_metrics['hurt_rate']:.3f}")
print(f"  Downstream F1 (Oracle Gate):  {val_eval_metrics['final_downstream_f1']:.4f}")
print(f"  Selector Headroom Recovery:   {val_eval_metrics['selector_headroom_recovery']*100:.1f}%")

# Compute calibration metrics
y_true = df_util_val["target_help_binary"].values
y_prob = np.clip(df_util_val["semantic_score"].values, 0.0, 1.0)
calib_res = compute_calibration_metrics(y_true, y_prob)
print(f"  Expected Calibration Error:   {calib_res['ece']:.4f}")
print(f"  Brier Score:                  {calib_res['brier_score']:.4f}")


## SECTION 18 — Save Frozen Selector Config


In [ ]:
# Strict scientific safeguard against writing uncomputed placeholder tables
if is_synthetic_fit:
    raise RuntimeError(
        "W8-C metrics have not been computed yet on real FIT. "
        "Do not write placeholder results."
    )

out_tables = REPO_ROOT / "results/week8/tables"
out_surfaces = REPO_ROOT / "results/week8/surfaces"
out_tables.mkdir(parents=True, exist_ok=True)
out_surfaces.mkdir(parents=True, exist_ok=True)

# Save verified summary table
summary_table.to_csv(out_tables / "w8c_utility_models_summary.csv", index=False)
calib_res["decile_table"].to_csv(out_tables / "w8c_selector_calibration_deciles.csv", index=False)

# Save frozen selector configuration for W8-E joint controller
import json
frozen_cfg = {
    "selected_model": best_selector_name,
    "internal_val_shr": val_eval_metrics["selector_headroom_recovery"],
    "internal_val_f1": val_eval_metrics["final_downstream_f1"],
    "mean_regret": val_eval_metrics["mean_regret"],
    "near_oracle_rate": val_eval_metrics["near_oracle_selection_rate"],
    "non_leaking_feature_columns": non_leaking_cols,
}
with open(out_surfaces / "w8c_frozen_selector_config.json", "w", encoding="utf-8") as f:
    json.dump(frozen_cfg, f, indent=2)

print(f"Successfully saved verified W8-C artifacts to {out_tables} and {out_surfaces}.")
